# Ingest into tables

This notebook reads the raw data downloaded and ingests it into tables in bronze

In [0]:
!ls /Volumes/bronze/methylation/geo_datasets

In [0]:
!ls /Volumes/bronze/methylation/geo_datasets/GSE213478/*
#!mv /Volumes/bronze/methylation/geo_datasets/GSE213478/GSE213478_part_0.txt.gz /Volumes/bronze/methylation/geo_datasets/GSE213478/part_0.txt.gz

In [0]:
!zcat /Volumes/bronze/methylation/geo_datasets/GSE213478/part_1.txt.gz | head -n 10

In [0]:
import pandas as pd



## Ingest Healthy data

In [0]:
%sql

-- DROP TABLE IF EXISTS bronze.methylation.GSE213478_beta

In [0]:
# Step 1: Read all .gz files in the folder as a single Spark DataFrame
df = spark.read \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .csv("dbfs:/Volumes/bronze/methylation/geo_datasets/GSE213478/")

# Step 2: Save as Spark SQL table
df.write \
    .mode("overwrite") \
    .format("delta") \
    .saveAsTable("bronze.methylation.GSE213478_beta")

print("✅ Table created: bronze.methylation.GSE213478_beta")


In [0]:
%sql

SELECT * FROM bronze.methylation.GSE213478_beta

## Pivot into long format

In [0]:
# GSE213478_beta

In [0]:
%sql
--DROP TABLE IF EXISTS bronze.methylation.GSE213478_beta_long

In [0]:
from pyspark.sql.functions import expr, col

# Step 1: Read files
df_wide = spark.read \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .csv("dbfs:/Volumes/bronze/methylation/geo_datasets/GSE213478/")

# Fix column name if needed
columns = df_wide.columns
if columns[0] == "_c0":
    df_wide = df_wide.withColumnRenamed("_c0", "probe_id")

# Step 2: Identify sample columns
sample_columns = [c for c in df_wide.columns if c != "probe_id"]

# Step 3: Build stack expression
stack_expr = f"stack({len(sample_columns)}, " + ", ".join(
    [f"'{c}', `{c}`" for c in sample_columns]
) + ") as (sample_id, beta)"

# Step 4: Melt to long format
df_long = df_wide.selectExpr("probe_id", stack_expr)

# Step 5: Write as Delta table
df_long.write \
    .mode("overwrite") \
    .format("delta") \
    .saveAsTable("bronze.methylation.GSE213478_beta_long")

print("✅ Table created: bronze.methylation.GSE213478_beta_long")


In [0]:
%sql

SELECT * FROM bronze.methylation.GSE213478_beta_long LIMIT 10

### Add variance and restrictions

In [0]:
df = spark.table("bronze.methylation.GSE213478_beta_long")  


In [0]:
from pyspark.sql.functions import lit

df = df.withColumn("access", lit("public"))
# Rename cpg_id to sample_id
#df = df.withColumnRenamed("cpg_id", "sample_id")


In [0]:
from pyspark.sql.functions import variance

# Compute variance of beta_value per sample (cpg_id)
var_df = df.groupBy("probe_id").agg(variance("beta").alias("probe_var"))

# Join back to original table
df = df.join(var_df, on="probe_id", how="left")


In [0]:
df.write.mode("overwrite").option("mergeSchema", "true").saveAsTable("bronze.methylation.GSE213478_beta_long")


In [0]:
%sql

--DROP TABLE IF EXISTS bronze.methylation.GSE289137_beta_long_with_variance;

SELECT * FROM bronze.methylation.GSE213478_beta_long LIMIT 10

In [0]:
#healthy_raw = pd.read_csv("/Volumes/bronze/methylation/geo_datasets/GSE213478/GPL21145_MethylationEPIC_15073387_v-1-0.csv.gz")
#healthy_raw
